In [1]:
# 1. Uninstall broken transformers build and reinstall compatible versions
!pip install --upgrade "transformers>=4.40.0,<4.45.0" "safetensors"

# 2. Re-run your PaddleOCR setup

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.2 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: 

In [2]:
%%writefile /kaggle/working/requirements.txt
# Base dependencies (Pinned for stability)
protobuf>=4.25.3,<6.0.0
opencv-python-headless

# Frameworks
paddlepaddle>=2.6.2
paddleocr>=2.7.3
transformers>=4.40.0,<4.45.0
paddlenlp>=2.8.0

Writing /kaggle/working/requirements.txt


In [3]:
# 1. Upgrade build tools
!pip install --upgrade pip setuptools wheel

# 2. Pre-install seqeval
!pip install seqeval --no-build-isolation

# 3. Install PaddlePaddle and PaddleOCR directly
!pip install paddlepaddle>=2.6.2 paddleocr>=2.7.3

# 3. Install requirements
!pip install --upgrade pip
!pip install --no-cache-dir -r /kaggle/working/requirements.txt

# 4. Clone repository
!git clone https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_det_safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/9d/2d/233c79d5b4e5ab1dbf111242299153f3caddddbb691219f363ad55ce783d/seqeval-1.2.2.tar.gz (from https://pypi.org/simple/seqeval/): Requested seqeval from https://files.pythonhosted.org/packages/9d/2d/233c79d5b4e5ab1dbf111242299153f3caddddbb691219f363ad55ce783d/seqeval-1.2.2.tar.gz has inconsistent version: expected '1.2.2', but metadata has '0.0.0'
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.

In [4]:
import google.protobuf
import paddle
import paddleocr
from transformers import AutoTokenizer

print("Protobuf version:", google.protobuf.__version__)
print("Paddle version:", paddle.__version__)
print("Setup complete successfully!")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Protobuf version: 5.29.5
Paddle version: 3.3.1
Setup complete successfully!


In [5]:
!ls -R /kaggle/working/PP-OCRv6_medium_det_safetensors

/kaggle/working/PP-OCRv6_medium_det_safetensors:
config.json    model.safetensors	 README.md
inference.yml  preprocessor_config.json


In [6]:
import json
from paddleocr import PaddleOCR

# 1. Initialize PaddleOCR
ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv6_medium_det",
    text_recognition_model_name="PP-OCRv6_medium_rec",
    # engine="transformers",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    enable_mkldnn=False,  # Bypasses the PIR / oneDNN CPU bug
    device="cpu"
)

# 2. Run prediction
img_url = "https://cdn-uploads.huggingface.co/production/uploads/681c1ecd9539bdde5ae1733c/3ul2Rq4Sk5Cn-l69D695U.png"
results = ocr.predict(img_url)

# 3. Extract text lines from results
for res in results:
    rec_texts = res.get('rec_texts', [])

    # Print lines to console
    full_text = "\n".join(rec_texts)
    print("--- EXTRACTED TEXT ---")
    print(full_text)

    # Save directly to a plain text file
    with open("extracted_text.txt", "w", encoding="utf-8") as f:
        f.write(full_text)

print("\nSaved extracted text to 'extracted_text.txt'")

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

inference.yml: 0.00B [00:00, ?B/s]

inference.pdiparams:   0%|          | 0.00/76.5M [00:00<?, ?B/s]

Connecting to https://cdn-uploads.huggingface.co/production/uploads/681c1ecd9539bdde5ae1733c/3ul2Rq4Sk5Cn-l69D695U.png ...
[==================================================] 100.00%


--- EXTRACTED TEXT ---
Algorithms for the Markov Entropy Decomposition
Andrew J. Ferris and David Poulin
Département de Physique, Université de Sherbrooke, Québec, J1K 2R1, Canada
(Dated: October 31, 2018)
The Markov entropy decomposition (MED) is a recently-proposed, cluster-based simulation method for fi-
nite temperature quantum systems with arbitrary geometry. In this paper, we detail numerical algorithms for
performing the required steps of the MED, principally solving a minimization problem with a preconditioned
rX V   012
Newton's algorithm, as well as how to extract global susceptibilities and thermal responses. We demonstrate
the power of the method with the spin-1/2 XXZ model on the 2D square lattice, including the extraction of
critical points and details of each phase. Although the method shares some qualitative similarities with exact-
diagonalization, we show the MED is both more accurate and significantly more flexible.
PACS numbers: 05.10.-a, 02.50.Ng, 03.67.-a, 74.40.K